# 第9课 · 每一帧声音都是一支箭——向量（vector）加法、缩放与线性组合（linear combination）

**学习目标**
1. 理解向量是 shape `(n,)` 的 NumPy array，`.shape` 和 `.dtype` 是必检属性
2. 掌握向量加法（混音）和标量缩放（调音量）的代数与几何含义
3. 实现 `scale(v, c)` 函数，通过单元测试验证标量乘法
4. 用 `np.linalg.norm` 计算向量长度，建立 MFCC 特征向量的维度直觉
5. 理解线性组合：任意 2D 向量可由基向量 `e1`, `e2` 加权叠加表示

**为什么对 Aurora 重要**：Aurora 的 MFCC 特征提取把每帧音频压缩成一个 `(13,)` 向量；`scale(v, c)` 调音量时就是对这个向量做标量缩放（scalar scaling）。神经网络每层的激活值同样是向量。

← **上一课**　[L08 · 复数平面可视化](../1_complex_trig/L08_visual_complex.ipynb)

> 上节课学习了 **复数平面可视化**：单位圆旋转、共轭与相位，matplotlib 动态演示。  
> 本课将探讨 **向量代数**。

## 本课剧情：一串数字，藏着方向和大小

你在混音台前，推一个推子——声音变大了。用数学说：每一个采样值乘以同一个系数，这就是**标量乘法**（scalar multiplication）。

再叠加两轨音频——男声和女声同时响。用数学说：两个等长数组逐元素相加，这就是**向量加法**（vector addition）。

向量的全部基本操作就是这两件事。一旦掌握它们：
- **矩阵 = 多个向量**（每列是一个方向）
- **特征向量 = 被矩阵拉伸但不旋转的向量**
- **SVD = 找到最重要的几个方向**

线代大厦的地基是向量。本课先练好两个"砖块"：`scale` 和 `add_signals`，再用它们拼出 `linear_combination`。

## 1. 向量就是 numpy 数组

有没有想过：为什么线性代数用到 NumPy？

因为 NumPy 的一维 array 就是向量——一个**有顺序的数列** `[x₁, x₂, …, xₙ]`，形状（shape）是 `(n,)`。

物理上，向量是"带方向的箭头"：
- 二维向量 `[3, 4]` → 从原点出发，向右 3、向上 4 的箭头，长度 5
- 音频信号 `[0.1, -0.3, 0.5, ...]` → N 维空间里的一个点（或一条路径）

音频处理中向量无处不在：
- 一帧 MFCC 系数 = 13 维向量（特征空间里的一个点）
- 一个词的词嵌入 = 768 维向量（语义空间里的一个点）
- 两帧音频之间的相似度 = 向量点积

**关键直觉**：`shape = (n,)` 是向量，`shape = (m, n)` 是矩阵（多个向量排列）。把 shape 读清楚，不会在矩阵乘法时迷路。

## 符号入口：先看形状，再看运算

线性代数里的对象都有明确形状：向量是 `(n,)`，矩阵是 `(m, n)`，矩阵乘向量会把 `(n,)` 变成 `(m,)`。每个例子都先标出输入、变换和输出。

In [ ]:
import numpy as np
v = np.array([3.0, 4.0])      # 2 维向量
audio = np.array([0.0, 0.5, 1.0, 0.5, 0.0])  # 5 个采样点 = 5 维向量
print('v =', v, '| 维度:', v.shape)
print('audio 是一个', audio.shape[0], '维向量')

## 动手观察：shape 和 norm

观察 `v.shape` 输出 `(2,)`、`A.shape` 输出 `(2, 2)`、`A @ v` 的结果——矩阵把向量拉伸后 shape 不变但值变了。`np.linalg.norm(v)` 返回 5.0，即 √(3²+4²)，这是向量长度的 NumPy 写法。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：任意向量都是基向量的线性组合

下面的代码展示**任意 2D 向量都是基向量 e₁、e₂ 的线性组合**：给定系数 (a, b)，输出向量就是 `a·e₁ + b·e₂`。这是线性组合（linear combination）最直接的几何含义——方向和长度由系数决定，基向量提供"坐标轴"。

> **注**：矩阵对向量的作用将在 L12（矩阵）中详细展开。

In [ ]:
import numpy as np

# 线性组合：3 个基向量用不同系数叠加
e1 = np.array([1.0, 0.0]); e2 = np.array([0.0, 1.0])
for a, b in [(1,0), (0,1), (2,-1), (0.5, 1.5)]:
    v = a*e1 + b*e2
    print(f'{a}*e1 + {b}*e2 = {v}')
print('→ 任意 2D 向量 = 系数1*e1 + 系数2*e2')


## 2. 加法 = 逐元素相加；缩放 = 每个元素乘同一个数

In [ ]:
a = np.array([1.0, 2.0])
b = np.array([3.0, 1.0])
print('a + b =', a + b)     # 信号叠加（混音）就是向量加法
print('2 * a =', 2 * a)     # 放大 2 倍 = 标量缩放

## 3. ✏️ 你的任务：实现 `scale`

`scale(v, c)` 把向量 `v` 的每个分量都乘以标量 `c`——这是音量旋钮的数学本质。

**推理路线**：
1. 标量缩放的几何意义：向量方向不变，长度变为原来的 `|c|` 倍；当 `c < 0` 时方向反转，`c = 0` 时变成零向量。
2. NumPy 广播：`c * v` 把标量 `c` 自动应用到 `v` 的每个元素——无需写 for 循环，底层是 SIMD 并行指令。
3. 返回 shape：`c * v` 的 shape 与 `v` 完全相同，不需要额外 reshape。

**参考输入输出**：`scale([1.0, 2.0], 2.0)` → `[2.0, 4.0]`；`scale([0.2, -0.4], 1.5)` → `[0.3, -0.6]`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `scale` 前明确三件事

- 输入：`v`（numpy array，任意 shape）；`c`（浮点数，缩放系数）
- 关键步骤：`c * v`，NumPy 的标量乘法对每个元素同等作用
- 返回：与 `v` 等 shape 的 array，每个元素值为原来的 `c` 倍

In [ ]:
def scale(v, c):
    # ✏️ TODO: 返回 c 倍的 v
    raise NotImplementedError("TODO: implement scale — hint: use c * v")


In [ ]:
try:
    audio = np.array([0.2, -0.4, 0.6, -0.8])
    louder = scale(audio, 1.5)
    print('原始:', audio)
    print('放大:', louder)
    assert np.allclose(louder, [0.3, -0.6, 0.9, -1.2]), '应是每个元素 ×1.5'
    print('\n✅ 通过：你会缩放向量了 = 调音量。')
except NotImplementedError as e:
    print(f'⚠️  还未实现：{e}')
    print('请完成 scale() 函数体，再重新运行此格。')


## ✏️ 练习 A：实现 `add_signals`（向量加法 = 混音）

向量加法的音频含义：把两路声音逐采样点叠加。
**推理路线**：`a[i] + b[i]` 对每个 i，等价于 NumPy 的 `a + b`。
- 输入：两个等长 numpy array `a`, `b`
- 输出：逐元素之和，shape 与输入相同

In [ ]:
def add_signals(a, b):
    # ✏️ TODO: 返回 a 与 b 的逐元素之和
    raise NotImplementedError("TODO: implement add_signals — hint: use a + b")


try:
    ch1 = np.array([0.1, 0.3, -0.2])
    ch2 = np.array([0.4, -0.1, 0.5])
    mix = add_signals(ch1, ch2)
    assert np.allclose(mix, [0.5, 0.2, 0.3]), '逐元素相加结果不对'
    assert mix.shape == ch1.shape, 'shape 必须与输入相同'
    print('✅ add_signals 通过')
except NotImplementedError as e:
    print(f'⚠️  还未实现：{e}')


## ✏️ 练习 B：实现 `linear_combination`（线性组合）

线性组合把一组 `(系数, 向量)` 对折叠成单一向量：
`coeffs[0]*vecs[0] + coeffs[1]*vecs[1] + … + coeffs[k-1]*vecs[k-1]`

- 输入：`coeffs`（长度 k 的浮点列表）、`vecs`（k 个等长 numpy array 的列表）
- 输出：一个 numpy array，shape 与 `vecs[0]` 相同

In [ ]:
def linear_combination(coeffs, vecs):
    # ✏️ TODO: 把 coeffs[i] * vecs[i] 全部加起来
    raise NotImplementedError("TODO: implement linear_combination — hint: use a running sum")


try:
    e1 = np.array([1.0, 0.0]); e2 = np.array([0.0, 1.0])
    # 2·e₁ + (-1)·e₂ 应等于 [2, -1]
    result = linear_combination([2, -1], [e1, e2])
    assert np.allclose(result, [2.0, -1.0]), '线性组合结果不对'
    # 零系数边界：0·e₁ + 0·e₂ = 零向量
    zero = linear_combination([0, 0], [e1, e2])
    assert np.allclose(zero, [0.0, 0.0]), '系数全为0应返回零向量'
    print('✅ linear_combination 通过')
except NotImplementedError as e:
    print(f'⚠️  还未实现：{e}')


## 4. 几何：向量是带方向的箭头

In [ ]:
import matplotlib.pyplot as plt
v = np.array([3.0, 4.0])
plt.figure(figsize=(4,4))
plt.quiver(0,0, v[0], v[1], angles='xy', scale_units='xy', scale=1)
plt.xlim(-1,5); plt.ylim(-1,6); plt.grid(True, alpha=.3)
plt.gca().set_aspect('equal'); plt.title('vector [3,4]'); plt.show()

**🔗 Aurora 连接**：`aurora/audio/mfcc.py` 中的 `mfcc()` 函数
返回形状为 `(n_frames, 13)` 的矩阵，每一行是一帧音频的 13 维 MFCC 系数向量；
取其中一行（如 `coeffs[i]`，shape `(13,)`）再用 `scale(frame, gain)` 做逐元素缩放，
就是 Aurora 音量归一化步骤的数学本质（调音量 = 乘标量，不改变频谱形状）。
下一课点积（L10）会让你定量比较两帧 MFCC 向量的相似度——语音检索和注意力机制的底层操作。

## 🎨 图示：向量 = 带方向的箭头，缩放 = 改变长度

In [ ]:
from aurora.laviz import style, arrows2d
style()
arrows2d([[3,4],[1,2],[6,8]], ['a=[3,4]','b=[1,2]','2a (缩放)'],
         title='向量与缩放');

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

1. 把 `A[0][1]` 从 `1.0` 改成 `0.0`——矩阵变成对角矩阵，x 分量不再受 y 影响；验证：只有非对角元素才会让不同分量互相影响。
2. 把 `A[0][1]` 改成负数（如 `-2.0`），先手算 `v=[1, 1]` 的输出，再运行对照——负值会把 x 分量"拉向反方向"。
3. 把 `probes` 里一个向量换成 `[3.0, 4.0]`（norm = 5.0），观察矩阵对"长"向量的缩放倍率是否与对 `[1, 1]` 相同（提示：矩阵变换一般**不**保持 norm 不变，除非是正交矩阵）。

## 本课收束

现在可以用 `scale(v, c)` 对任意 numpy array 做逐元素缩放，输出 shape 与输入完全相同。这对应 Aurora `aurora/audio` 里的调音量操作，而向量加法对应混音——两个操作合起来覆盖了信号合成的底层逻辑。MFCC 特征提取最终把每帧压成一个 `(13,)` 向量，就是今天操作的对象形式。下一节点积会让你计算两个向量的相似度，这是特征匹配和注意力机制的基础运算。

## ✏️ 白板挑战：向量运算手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：v = [3, 4]，c = 2.5，`scale(v, c)` = ?

**问 2**：a = [1.0, -0.5, 0.8]，b = [0.2, 0.7, -0.3]，`add_signals(a, b)` = ?

**问 3**：coeffs = [2.0, -1.0, 0.5]，vecs = [[1,0],[0,1],[2,2]]  
`linear_combination(coeffs, vecs)` = 2×[1,0] + (−1)×[0,1] + 0.5×[2,2] = ?

**问 4**：向量 [3, 4] 的 L2 范数（欧式长度）是多少？公式：`‖v‖ = √(v₁² + v₂²)` = ?

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

passed = []

# 问1：scale
v, c = np.array([3.0, 4.0]), 2.5
q1_expected = np.array([7.5, 10.0])
try:
    q1 = scale(v, c)
    assert np.allclose(q1, q1_expected, atol=1e-10), f"scale 结果 {q1} 与期望 {q1_expected} 不符"
    print(f"Q1 ✅  scale([3,4], 2.5) = {q1}")
    passed.append(1)
except NotImplementedError:
    print("⬜ Q1：请先实现 scale()，再运行对答案格")

# 问2：add_signals
a, b = np.array([1.0, -0.5, 0.8]), np.array([0.2, 0.7, -0.3])
q2_expected = np.array([1.2, 0.2, 0.5])
try:
    q2 = add_signals(a, b)
    assert np.allclose(q2, q2_expected, atol=1e-10)
    print(f"Q2 ✅  add_signals = {q2}")
    passed.append(2)
except NotImplementedError:
    print("⬜ Q2：请先实现 add_signals()，再运行对答案格")

# 问3：linear_combination
coeffs = [2.0, -1.0, 0.5]
vecs   = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([2.0, 2.0])]
q3_expected = np.array([3.0, 0.0])
try:
    q3 = linear_combination(coeffs, vecs)
    assert np.allclose(q3, q3_expected, atol=1e-10)
    print(f"Q3 ✅  linear_combination = {q3}")
    passed.append(3)
except NotImplementedError:
    print("⬜ Q3：请先实现 linear_combination()，再运行对答案格")

if len(passed) == 3:
    print("\n🎉 白板挑战通过！向量三运算已内化。")


In [ ]:
# ✏️ 本课自评
l09_review = {
    "scale_implemented":             None,  # scale 实现并通过断言？True/False
    "add_signals_implemented":       None,  # add_signals 实现并通过断言？True/False
    "linear_combination_implemented": None, # linear_combination 实现并通过断言？True/False
    "vector_as_arrow_intuition":     None,  # 理解"向量=带方向的箭头"？True/False
    "whiteboard_passed":             None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l09_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l09_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L09 全部通关！进入 L10：点积与投影')

---

→ **下一课**　[L10 · 点积与投影](L10_dot_product.ipynb)

> 下节课将学习 **点积与投影**：a·b = |a||b|cosθ，为什么相似度 = 点积 ÷ 积范数。